In [1]:
import peyes
import pandas as pd
import pyplnoise
from pathlib import Path
from test_tools import *
from acma import acma
import sys

# Init Detectors

In [2]:
base_params = {
    "missing_value": np.nan,
    "min_event_duration": 10,
    "pad_blinks_time": 10
}

det_ivt = peyes.create_detector(
    algorithm="ivt",
    **base_params,
    saccade_velocity_threshold=50.0
)

det_ivvt = peyes.create_detector(
    algorithm="ivvt",
    **base_params,
    saccade_velocity_threshold=50.0,
    smooth_pursuit_velocity_threshold=20.0
)

det_idt = peyes.create_detector(
    algorithm="idt",
    **base_params,
    dispersion_threshold=0.4,
    window_duration=20
)

det_idvt = peyes.create_detector(
    algorithm="idvt",
    **base_params,
    dispersion_threshold=0.2,
    window_duration=30,
    saccade_velocity_threshold=60.0
)

det_engbert = peyes.create_detector(
    algorithm="engbert",
    **base_params,
    lambda_param=3.0,
    deriv_window_size=5
)

det_nh = peyes.create_detector(
    algorithm="nh",
    **base_params,
    filter_duration_ms=20,
    filter_polyorder=2,
    saccade_max_velocity=1000.0,
    saccade_max_acceleration=100000.0,
    min_saccade_duration=10,
    min_fixation_duration=40,
    max_pso_duration=40
)

det_remodnav = peyes.create_detector(
    algorithm="remodnav",
    **base_params,
    min_saccade_duration=10,
    min_smooth_pursuit_duration=40,
    min_fixation_duration=40,
    min_blink_duration=20,
    max_pso_duration=40,
    median_filter_duration_ms=50
)

def i2mc_detector(t, x, y):
    import I2MC
    gazeData = pd.DataFrame({'time': t, 'L_X': x, 'L_Y': y})
    sr = 1000.0 / np.mean(np.diff(t))
    sr = round(sr, 1)

    opt = {
        'xres': 1920.0,
        'yres': 1080.0,
        'missingx': np.nan,
        'missingy': np.nan,
        'freq': sr,
        'downsamples': [2, 5],
        'downsampFilter': True,
        'cutoffstd': 2,
        'onoffsetThresh': 3,
        'maxMergeDist': 30.0,
        'maxMergeTime': 30.0,
        'minFixDur': 40.0
    }

    i2mc_fix, i2mc_data, _ = I2MC.I2MC(gazeData, options=opt)

    labels = np.full(len(t), 'sac', dtype=object)

    missing_mask = np.isnan(x) | np.isnan(y)
    labels[missing_mask] = 'blink'

    if i2mc_fix is not False and i2mc_fix is not None:
        starts = i2mc_fix['start']
        ends = i2mc_fix['end']

        for s, e in zip(starts, ends):
            labels[s:e+1] = 'fix'

    return labels


def uneye(t, x, y, l, dataset=None):
    import uneye

    repo_root = Path.cwd()
    weights_dir_demo = repo_root / "training"

    dt = np.nanmedian(np.diff(t))
    sampfreq = int(round(1000.0 / dt))
    print(sampfreq)

    min_sacc_dur = 6
    min_sacc_dist = 10

    label_map = {"fix": 0,"sac": 1,}

    labels = np.full(l.shape, np.nan, dtype=float)
    for key, value in label_map.items():
        labels[l == key] = value

    valid = np.isfinite(x) & np.isfinite(y)
    x_valid = x[valid]
    y_valid = y[valid]
    l_valid = labels[valid].astype(int)

    trial_len = int(sampfreq)
    n_trials = x_valid.size // trial_len

    x_valid = x_valid[: n_trials * trial_len]
    y_valid = y_valid[: n_trials * trial_len]
    l_valid = l_valid[: n_trials * trial_len]

    X = x_valid.reshape(n_trials, trial_len)
    Y = y_valid.reshape(n_trials, trial_len)
    L = l_valid.reshape(n_trials, trial_len)

    if dataset == 'eyelink':
        weights_name = str(weights_dir_demo / "weights_eyelink")
    elif dataset == 'sim':
        weights_name = str(weights_dir_demo / "weights_sim")
    elif dataset == 'andersson':
        weights_name = str(weights_dir_demo / "weights_andersson")
    else:
        print('Error load weights')
        return None

    model = uneye.DNN(weights_name=weights_name,
             sampfreq=sampfreq,
             min_sacc_dur=min_sacc_dur,
            min_sacc_dist=min_sacc_dist)

    pred, prob = model.predict(X,Y)

    pred_all_1d = pred.reshape(-1)
    pred_full = np.full(len(l), np.nan)
    valid_idx = np.where(valid)[0][: pred_all_1d.size]
    pred_full[valid_idx] = pred_all_1d

    inv_label_map = {v: k for k, v in label_map.items()}

    pred_labels = np.full(len(l), "blink", dtype=object)
    ok = np.isfinite(pred_full)
    codes = pred_full[ok].astype(np.int64, copy=False)
    pred_labels[ok] = [inv_label_map[int(c)] for c in codes]

    return pred_labels


def acma_detector(t, x, y, method='pca'):
    sr = 1000.0 / np.mean(np.diff(t))
    sr = round(sr, 1)
    acma_data = acma(x, y, freq=sr, n_delay=3, r2_th=0.8, method=method,window_time=0.3)

    return acma_data['label'], acma_data['raw_mask']


# Benchmark

In [ ]:
import numpy as np
import pandas as pd
import pyplnoise
from functools import partial
from sklearn.metrics import cohen_kappa_score

FREQ              = 1000.0
ps, vd            = 0.037, 67
TARGET_CLASSES    = ['sac', 'fix']
overlap_threshold = 0.25
RMS_LEVELS        = list(range(1, 40, 2))
DATASETS          = ['andersson', 'eyelink', 'sim']

def scale_to_rms(n, d):
    return n * (d / np.sqrt(np.mean(n**2)))

def make_noisy(x, y, rms):
    ng_x = pyplnoise.AlphaNoise(FREQ, 1e-3, FREQ/2., alpha=1)
    ng_y = pyplnoise.AlphaNoise(FREQ, 1e-3, FREQ/2., alpha=1)
    return (x + scale_to_rms(ng_x.get_series(len(x)), rms),
            y + scale_to_rms(ng_y.get_series(len(y)), rms))

def kappa_for(labels, true_lbl, tc):
    y_true = (np.asarray(true_lbl) == tc).astype(int)
    y_pred = (np.asarray(labels)   == tc).astype(int)
    return cohen_kappa_score(y_true, y_pred)

def run_acma(t, x, y, **ctx):
    labels, _ = acma_detector(t, x, y, method='pca')
    return labels

def run_i2mc(t, x, y, **ctx):
    return i2mc_detector(t, x, y)

def run_peyes(detector, t, x, y, **ctx):
    labels, _ = detector.detect(t=t, x=x, y=y,
                                pixel_size_cm=ps, viewer_distance_cm=vd)
    return unified_peyes_label(labels)

def run_uneye(t, x, y, true_lbl=None, dataset=None, **ctx):
    return uneye(t, x, y, true_lbl, dataset=dataset)

DETECTORS = {
    'ACMA':     run_acma,
    'I2MC':     run_i2mc,
    'NH':       partial(run_peyes, det_nh),
    'REMODNAV': partial(run_peyes, det_remodnav),
    'IVT':      partial(run_peyes, det_ivt),
    'IVVT':     partial(run_peyes, det_ivvt),
    'IDT':      partial(run_peyes, det_idt),
    'IDVT':     partial(run_peyes, det_idvt),
    'Engbert':  partial(run_peyes, det_engbert),
    'UnEye':    run_uneye,
}

def run_task(dataset, rms, name, fn, t_ms, x, y, true_lbl):
    metrics = {
        'F1':        {tc: np.nan for tc in TARGET_CLASSES},
        'Kappa':     {tc: np.nan for tc in TARGET_CLASSES},
        'Onset_ms':  {'sac': np.nan},
        'Offset_ms': {'sac': np.nan},
    }
    try:
        labels = fn(t_ms, x, y, true_lbl=true_lbl, dataset=dataset)
        labels = np.asarray(labels)

        for tc in TARGET_CLASSES:
            _, _, m = evaluate_event_level_1to1_2(
                true_lbl, labels, target_class=tc,
                overlap_threshold=overlap_threshold,
                t_ms=t_ms if tc == 'sac' else None,
                verbose=False,
            )
            metrics['F1'][tc]    = m['f1']
            metrics['Kappa'][tc] = kappa_for(labels, true_lbl, tc)
            if tc == 'sac':
                metrics['Onset_ms']['sac']  = m['onset_ms']
                metrics['Offset_ms']['sac'] = m['offset_ms']
    except Exception as e:
        print(f"[!] {name} @ ds={dataset}, RMS={rms} failed: {type(e).__name__}: {e}")
    return dataset, rms, name, metrics

all_results = []
for dataset in DATASETS:
    print(f"\n>>> Dataset: {dataset}")
    eye_data = pd.read_csv(f'./datasets/{dataset}_data.csv')

    if dataset == 'eyelink':
        start_idx, end_idx = 0, 500000
    elif dataset == 'sim':
        start_idx, end_idx = 0, eye_data.shape[0]//2
    elif dataset == 'andersson':
        start_idx, end_idx = 0, eye_data.shape[0]

    t_slice  = eye_data['t'].to_numpy(dtype=float)[start_idx:end_idx]
    x_clean  = eye_data['x'].to_numpy(dtype=float)[start_idx:end_idx]
    y_clean  = eye_data['y'].to_numpy(dtype=float)[start_idx:end_idx]
    true_lbl = eye_data['label'].astype(str).str.lower().to_numpy()[start_idx:end_idx]
    t_ms     = t_slice * 1000.0

    noisy_signals = {r: make_noisy(x_clean, y_clean, r) for r in RMS_LEVELS}

    for rms in RMS_LEVELS:
        x_noisy, y_noisy = noisy_signals[rms]
        for name, fn in DETECTORS.items():
            print(f"  RMS={rms}, Detector={name}")
            result = run_task(dataset, rms, name, fn, t_ms, x_noisy, y_noisy, true_lbl)
            all_results.append(result)

rows = []
for dataset, rms, name, metrics in all_results:
    for metric_name, by_class in metrics.items():
        for tc, val in by_class.items():
            rows.append((dataset, rms, name, tc, metric_name, val))
df_long = pd.DataFrame(rows,
    columns=['dataset', 'RMS_Noise', 'Detector', 'target_class', 'metric', 'value'])

df_results = (
    df_long
    .pivot_table(index=['dataset', 'RMS_Noise', 'target_class', 'metric'],
                 columns='Detector', values='value')
    .reindex(columns=list(DETECTORS.keys()))
    .reset_index()
)

to_mean = df_results[
    df_results['metric'].isin(['F1', 'Kappa']) &
    df_results['target_class'].isin(['sac', 'fix'])
]
mean_rows = (
    to_mean
    .groupby(['dataset', 'RMS_Noise', 'metric'])[list(DETECTORS.keys())]
    .mean()
    .reset_index()
    .assign(target_class='mean')
)
df_results = pd.concat([df_results, mean_rows], ignore_index=True)

tc_order     = {'sac': 0, 'fix': 1, 'mean': 2}
metric_order = {'F1': 0, 'Kappa': 1, 'Onset_ms': 2, 'Offset_ms': 3}
ds_order     = {ds: i for i, ds in enumerate(DATASETS)}
df_results = (
    df_results
    .assign(_d=df_results['dataset'].map(ds_order),
            _t=df_results['target_class'].map(tc_order),
            _m=df_results['metric'].map(metric_order))
    .sort_values(['_d', 'RMS_Noise', '_t', '_m'])
    .drop(columns=['_d', '_t', '_m'])
    .reset_index(drop=True)
)

df_results = df_results[
    ['dataset', 'RMS_Noise', 'target_class', 'metric'] + list(DETECTORS.keys())
]
df_results.to_csv("results_metrics.csv", index=False, encoding="utf-8")
display(df_results)

# Hyperparameter optimization

In [ ]:
from joblib import Parallel, delayed
import pyplnoise, peyes, numpy as np, pandas as pd

RMS = 1
ng_x = pyplnoise.AlphaNoise(FREQ, 1e-3, FREQ/2., alpha=1)
ng_y = pyplnoise.AlphaNoise(FREQ, 1e-3, FREQ/2., alpha=1)
x_n = x_clean + scale_to_rms(ng_x.get_series(len(x_clean)), RMS)
y_n = y_clean + scale_to_rms(ng_y.get_series(len(y_clean)), RMS)

ivt_grid = [dict(saccade_velocity_threshold=float(v))
            for v in range(5, 101, 10)]

ivvt_grid = [dict(saccade_velocity_threshold=float(sv),
                  smooth_pursuit_velocity_threshold=float(pv))
             for sv in range(10, 101, 5)
             for pv in range(1, 31, 2) if pv < sv]

idt_grid = [dict(dispersion_threshold=round(d, 2), window_duration=int(w))
            for d in np.arange(0.2, 3.01, 0.2)
            for w in range(20, 101, 10)]

idvt_grid = [dict(dispersion_threshold=round(d, 2),
                  window_duration=int(w),
                  saccade_velocity_threshold=float(sv))
             for d  in np.arange(0.2, 2.51, 0.2)
             for w  in range(30, 101, 10)
             for sv in range(20, 101, 20)]

engbert_grid = [dict(lambda_param=round(lp, 2), deriv_window_size=int(dw))
                for lp in np.arange(2.0, 10.01, 1.0)
                for dw in [2, 3, 4, 5, 6]]

def eval_params(algo, params):
    det = peyes.create_detector(algorithm=algo, **base_params, **params)
    labels, _ = det.detect(t=t_ms, x=x_n, y=y_n,
                           pixel_size_cm=ps, viewer_distance_cm=vd)
    labels = unified_peyes_label(labels)
    _, _, m = evaluate_event_level_1to1_2(
        true_lbl, labels, target_class='sac',
        overlap_threshold=0.5, t_ms=t_ms, verbose=False
    )
    return algo, params, m['f1'], m['onset_ms'], m['offset_ms']

tasks = ([('ivt',     p) for p in ivt_grid]     +
         [('ivvt',    p) for p in ivvt_grid]    +
         [('idt',     p) for p in idt_grid]     +
         [('idvt',    p) for p in idvt_grid]    +
         [('engbert', p) for p in engbert_grid])

results = Parallel(n_jobs=-1, backend='loky', verbose=5)(
    delayed(eval_params)(a, p) for a, p in tasks
)

df = pd.DataFrame(results, columns=['algo', 'params', 'F1', 'Onset_ms', 'Offset_ms'])
best = df.loc[df.groupby('algo')['F1'].idxmax()].set_index('algo')

for algo in ['ivt', 'ivvt', 'idt', 'idvt', 'engbert']:
    row = best.loc[algo]
    print(f"{algo.upper():8s}  F1={row['F1']:.4f}  "
          f"Δonset={row['Onset_ms']:.1f}ms  Δoffset={row['Offset_ms']:.1f}ms  "
          f"{row['params']}")